In [ ]:
!pip install statsforecast scikit-learn carbontracker eco2ai setuptools

In [ ]:
import time
import numpy as np
import pandas as pd
import json

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, Naive, SeasonalNaive
from sklearn.metrics import mean_absolute_error, mean_squared_error

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions.csv", ignore_warnings=True)

In [ ]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

In [ ]:
def load_series(path, unique_id):
    df = pd.read_csv(path, index_col='date')
    df = df.reset_index().rename(columns={'date': 'ds', 'value': 'y'})
    df['unique_id'] = unique_id
    return df

def smape(y_true, y_pred, decimals: int = 2) -> float:
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return round(float(np.mean(numerator / (denominator + 1e-10)) * 100), decimals)

def run_forecast(train_df, true_df, freq, season_length, dataset_name):
  results = []

  models = [
    ("AutoARIMA", AutoARIMA()),
    ("AutoETS", AutoETS(season_length=season_length)),
    ("Naive", Naive()),
    ("SeasonalNaive", SeasonalNaive(season_length=season_length)),
  ]

  for model_name, model in models:
    sf = StatsForecast(
      models=[model],
      freq=freq,
      n_jobs=1
    )

    eco2ai_tracker.start()

    with CarbonTracker() as carbontracker:
      start = time.time()
      forecast = sf.forecast(df=train_df, h=len(true_df))
      end = time.time()

    eco2ai_tracker.stop()

    y_true = true_df['y'].values
    y_pred = forecast[model_name].values

    carbontracker_g = carbontracker.co2_g
    carbontracker_kWh = carbontracker.energy_kwh
    carbontracker_Wh = carbontracker_kWh * 1000 if carbontracker_kWh else None

    eco2ai_data = pd.read_csv("eco2ai_emissions.csv").iloc[-1]
    eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
    eco2ai_Wh = eco2ai_kWh * 1000
    eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
    eco2ai_g = eco2ai_kg * 1000

    results.append({
      "Dataset": dataset_name,
      "Modelo": model_name,
      "Parâmetros": str(model),
      "SMAPE": smape(y_true, y_pred),
      "MAE": mean_absolute_error(y_true, y_pred),
      "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
      "CarbonTracker CO₂ (g)": carbontracker_g,
      "CarbonTracker Consumo (Wh)": carbontracker_Wh,
      "Eco2AI CO₂ (g)": eco2ai_g,
      "Eco2AI Consumo (Wh)": eco2ai_Wh,
      "Tempo (s)": end - start,
      "y_true": json.dumps(y_true.tolist()),
      "y_pred": json.dumps(y_pred.tolist())
    })

  return results

In [ ]:
etth2 = load_series('data/ETTh2.csv', 'etth2')
electricity = load_series('data/Electricity.csv', 'electricity')
traffic = load_series('data/Transporte.csv', 'traffic')
covid19 = load_series('data/Covid-19.csv', 'covid19')
mortality = load_series('data/Mortalidade.csv', 'mortality')
retail = load_series('data/Varejo.csv', 'retail')
wike2000 = load_series('data/Wike2000.csv', 'wike2000')

In [ ]:
etth2_train = etth2[(etth2['ds'] >= '2016-07-12 06:00:00') & (etth2['ds'] <= '2016-09-20 05:00:00')]
etth2_true = etth2[etth2['ds'] > '2016-09-20 05:00:00'].iloc[:24]

electricity_train = electricity[(electricity['ds'] >= '2013-06-28 07:00:01') & (electricity['ds'] <= '2013-09-06 06:00:01')]
electricity_true = electricity[electricity['ds'] > '2013-09-06 06:00:01'].iloc[:24]

traffic_train = traffic[(traffic['ds'] >= '2018-04-06 14:00:00') & (traffic['ds'] <= '2018-06-15 13:00:00')]
traffic_true = traffic[traffic['ds'] > '2018-06-15 13:00:00'].iloc[:24]

covid19_train = covid19[(covid19['ds'] >= '2021-03-11') & (covid19['ds'] <= '2022-03-09')]
covid19_true = covid19[covid19['ds'] > '2022-03-09'].iloc[:7]

mortality_train = mortality[(mortality['ds'] >= '2021-02-23') & (mortality['ds'] <= '2022-02-21')]
mortality_true = mortality[mortality['ds'] > '2022-02-21'].iloc[:7]

retail_train = retail[(retail['ds'] >= '2017-01-01') & (retail['ds'] <= '2017-12-30')]
retail_true = retail[retail['ds'] > '2017-12-30'].iloc[:7]

wike2000_train = wike2000[(wike2000['ds'] >= '2012-07-08') & (wike2000['ds'] <= '2013-07-06')]
wike2000_true = wike2000[wike2000['ds'] > '2013-07-06'].iloc[:7]

In [ ]:
all_results = []

all_results += run_forecast(etth2_train, etth2_true, freq='h', season_length=24, dataset_name='ETTh2')
all_results += run_forecast(electricity_train, electricity_true, freq='h', season_length=24, dataset_name='Electricity')
all_results += run_forecast(traffic_train, traffic_true, freq='h', season_length=24, dataset_name='Transporte')
all_results += run_forecast(covid19_train, covid19_true, freq='d', season_length=7, dataset_name='Covid-19')
all_results += run_forecast(mortality_train, mortality_true, freq='d', season_length=7, dataset_name='Mortalidade')
all_results += run_forecast(retail_train, retail_true, freq='d', season_length=7, dataset_name='Varejo')
all_results += run_forecast(wike2000_train, wike2000_true, freq='d', season_length=7, dataset_name='Wike2000')

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv("experimentos_estatisticos.csv", index=False)
results_df